# Допуск: SkillsBench: Benchmarking How Well Agent Skills Work Across Diverse Tasks

**Статья:** X. Li, Y. Liu, W. Chen, B. You и др., 2026 — <https://arxiv.org/abs/2602.12670>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций заменены: определите, что чему соответствует, и
  ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его;
- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы, опираясь на код и статью.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / рисунок / приложение / цитата) нарушено и как должно быть. Код писать не
нужно — достаточно объяснить словами. Запускать и отлаживать код тоже не нужно: все задания
решаются чтением кода и статьи.

Общие импорты:

In [ ]:
from statistics import mean
import torch
import yaml

`task` — задача SkillsBench: `task.instruction` — текст задания (тело `task.md`),
`task.skills` — curated Skills задачи (каталог `environment/skills/`), а также окружение,
эталонное решение (oracle) и верификатор. `agent.solve(task, skills=...)` — один прогон агента
(модель + harness) в свежем контейнере с указанными Skills; возвращает результат, который
оценивает `verify(task, out)` / `run_verifier(task, out)` — награда верификатора r ∈ [0, 1].
Функции, которые используются, но не показаны, считайте корректными.

## Задание 1. Пять условий эксперимента

**Тип задания:** что есть что — названия функций заменены, определите, что чему соответствует.

Каждая функция возвращает награды трёх прогонов одной задачи в одном условии. Среди них:
без Skills; с curated Skills; две реализации условия self-generated Skills (так, как оно было
устроено у Claude Code и у Codex / Gemini CLI, прил. D.6); и одно условие, которое статья
предлагает, но не оценивает. Определите, какая функция что реализует. Ошибок в коде нет.

In [ ]:
def run_arm(task, agent, skills, n_trials=3):
    return [verify(task, agent.solve(task, skills=skills)) for _ in range(n_trials)]

def arm_a(task, agent, n_trials=3):
    rewards = []
    for _ in range(n_trials):
        sandbox = new_sandbox(task, skills=[SKILL_CREATOR])
        agent.run(sandbox, CREATOR_PROMPT)                       # writes /app/generated-skills/*
        packs = sandbox.read_dir("/app/generated-skills")
        rewards.append(verify(task, agent.solve(task, skills=packs, sandbox=sandbox)))  # new session
    return rewards

def arm_b(task, agent):
    return run_arm(task, agent, skills=[])

def arm_c(task, agent):
    return run_arm(task, agent, skills=random_text_like(task.skills))   # same token length

def arm_d(task, agent):
    return run_arm(task, agent, skills=task.skills)

def arm_e(task, agent, n_trials=3):
    sandbox = new_sandbox(task, skills=[SKILL_CREATOR])
    agent.run(sandbox, CREATOR_PROMPT)                           # writes /app/generated-skills/*
    packs = sandbox.read_dir("/app/generated-skills")
    return run_arm(task, agent, skills=packs, n_trials=n_trials)

**Вопросы.**
1. По табл. 6 self-generated Skills на всех трёх конфигурациях оказываются *ниже* условия
   без Skills. Какие механизмы провала нашёл аудит траекторий (прил. D.6.1)? Какие из них
   возможны только в одной из двух реализаций (`arm_a` или `arm_e`) и почему? Почему случай
   `fix-visual-stability` (self-generated 1.0 против curated 0.0) не говорит, что
   самодельные Skills лучше?
2. Какая функция реализует условие, которого в экспериментах статьи нет? Какой вопрос оно
   должно закрыть (§6.1) и почему условие self-generated отвечает на него лишь частично?
3. Все условия запускаются на той же задаче, в том же контейнере и с тем же верификатором, а
   выигрыш считается как разность с одним и тем же baseline (рис. 3). Что дают такие парные
   разности по сравнению со сравнением «пулов» прогонов? Какие ещё отличия, кроме содержания
   Skills, между столбцами Self-Gen и No Sk. в табл. 6 называет сама статья?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Как Skills попадают к агенту

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Harness собирает стартовый контекст агента в условиях без Skills (`condition = "none"`) и с
curated Skills (`"curated"`) и по траектории определяет, обращался ли агент к Skills задачи
(Skill Invocation Rate, рис. 5, прил. K). `SKILLS_ROOT` — место, куда смонтирован каталог
`environment/skills/` задачи. Найдите одно отклонение от статьи.

In [ ]:
def parse_skill(skill_dir):
    text = (skill_dir / "SKILL.md").read_text()
    _, front, body = text.split("---", 2)
    meta = yaml.safe_load(front)                                    # YAML frontmatter
    return {"name": meta["name"], "description": meta["description"],
            "path": f"{SKILLS_ROOT}/{skill_dir.name}", "body": body}

def build_context(task, condition):
    skills = ([parse_skill(d) for d in sorted(task.skills_dir.iterdir())]
              if condition == "curated" else [])
    system = HARNESS_SYSTEM_PROMPT
    if skills:
        menu = "\n".join(f"- {s['name']}: {s['description']} ({s['path']}/SKILL.md)" for s in skills)
        system += "\n\nAvailable skills (read SKILL.md and bundled files on demand):\n" + menu
    instruction = task.instruction
    if skills:
        instruction += "\n\nUse these skills: " + ", ".join(s["name"] for s in skills)
    return [{"role": "system", "content": system}, {"role": "user", "content": instruction}]

def skill_invoked(trajectory):
    # counts only the task's own Skills; harness-bundled Skills are part of the harness
    return any(step.tool in ("read_file", "bash") and SKILLS_ROOT in step.arg
               for step in trajectory)

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Приём задачи в бенчмарк

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Путь присланной задачи от PR до ручного ревью (§3, прил. B, M). `extract_specifics` достаёт
из oracle, верификатора и окружения задачи то, что специфично для этого экземпляра: имена
файлов, пути, «магические» числа, команды эталонного решения, ожидаемые ответы.

In [ ]:
LEAK_FIELDS = ("filenames", "paths", "magic_numbers", "oracle_commands", "expected_outputs")

def admit(task, agents, n_trials=3, min_gap=0.05):
    if not structure_ok(task):                       # task.md schema, Dockerfile, oracle/, verifier/
        return "reject: structure"
    if run_verifier(task, run_oracle(task)) < 1.0:   # the oracle must pass every check
        return "reject: oracle"
    if not human_authored(task.instruction):         # AI-text detector + human label
        return "reject: instruction provenance"
    banned = extract_specifics(task.oracle, task.verifier, task.environment)
    for f in task.skill_files():
        if any(tok in f.text for field in LEAK_FIELDS for tok in banned[field]):
            return "reject: skill-to-solution leakage"
    gaps = []
    for agent in agents:                             # reviewers run several agents, both conditions
        no = mean(run_verifier(task, agent.solve(task, skills=[])) for _ in range(n_trials))
        cur = mean(run_verifier(task, agent.solve(task, skills=task.skills)) for _ in range(n_trials))
        gaps.append(cur - no)
    if max(gaps) < min_gap:                          # no measurable separation between conditions
        return "reject: low signal"
    return "maintainer review"

**Вопросы.**
1. Зачем требовать, чтобы oracle проходил верификатор на 100 %, и почему верификатор —
   детерминированный `pytest`, а не LLM-судья, хотя стандарт `task.md` его допускает
   (табл. 4)? Что сломалось бы в парном сравнении с LLM-судьёй? Приведите пример из
   статьи, где ошибся сам верификатор.
2. Почему проверка на утечку названа центральной для парного дизайна (§3, прил. M.7): что
   измерял бы Δ, если бы Skill содержал команды эталонного решения? Skill и oracle задачи
   часто содержат одинаковые строки вроде `from statsmodels.tsa.filters.hp_filter import
   hpfilter` или одни и те же числовые коэффициенты из стандартной формулы предметной
   области. Это утечка? Почему буквального совпадения недостаточно и что ещё, кроме этого
   фильтра, защищает от утечки?
3. Последний фильтр отбрасывает задачи, где Skills не дают разницы, а критерии качества
   (прил. B.10, A.4) требуют задач, которые «significantly easier with Skills», и Skills из
   верхнего квартиля по качеству. Как это влияет на то, что означает +16.6 п.п. из табл. 2?
   Можно ли ждать такого же выигрыша от Skills «из экосистемы»?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Pass rate и нормализованный выигрыш

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Метрики табл. 2 для C конфигураций «модель + harness». Найдите одно отклонение от §4
(Metrics) и прил. N.4.

In [ ]:
def skill_efficacy(no_skill, curated):
    # no_skill, curated: (C, 87, 3) verifier rewards in [0, 1]; NaN = missing result slot
    def pass_rate(r):
        r = torch.nan_to_num(r, nan=0.0)              # fixed 87 x 3 frame: a missing trial is a fail
        return r.mean(dim=2).mean(dim=1)              # trials within a task, then 87 tasks -> (C,)
    p0, p1 = pass_rate(no_skill), pass_rate(curated)
    delta = p1 - p0                                   # paired: same tasks, same containers
    g = delta / p0                                    # normalized gain
    summary = {"no": p0.mean(), "with": p1.mean(), "delta": delta.mean(),
               "g": g.mean()}                         # Mean row: macro-average of per-config g
    return p0, p1, delta, g, summary

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Анализ по доменам и устройству Skills

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

`p0_task`, `p1_task` — pass rate каждой задачи без Skills и с curated Skills для каждой из 18
конфигураций, формы `(18, 87)`. Функции строят табл. 3, 8, 9 (§5.1.2, прил. F). Найдите один
неверный комментарий.

In [ ]:
def lift_table(p0_task, p1_task, group_of):
    # group_of: 87 labels (domain, Skill-count bucket or complexity bucket)
    out = {}
    for g in sorted(set(group_of)):
        idx = torch.tensor([i for i, x in enumerate(group_of) if x == g])
        no, cur = p0_task[:, idx].mean(), p1_task[:, idx].mean()    # macro over tasks and configs
        out[g] = (len(idx), no.item(), cur.item(), (cur - no).item())
    return out

def count_bucket(n_skills):
    return "1" if n_skills == 1 else "2-3" if n_skills <= 3 else ">=4"

def complexity_bucket(skill_md_bytes, cuts):
    # cuts: 25th, 50th and 95th percentiles of total SKILL.md size over the 87 tasks
    q25, q50, q95 = cuts
    return ("compact" if skill_md_bytes <= q25 else "standard" if skill_md_bytes <= q50
            else "detailed" if skill_md_bytes <= q95 else "comprehensive")

# by domain: every domain gains; domains with procedures underrepresented in pretraining
# (Natural Science, Media, Cybersecurity) gain most, Software Engineering and Math & OR least
# by count and complexity: more Skills and longer SKILL.md carry more procedural knowledge,
# so the lift grows monotonically: >=4 Skills and comprehensive bundles gain the most

**Ваш ответ:**

_Напишите ответ здесь._